# Experiment 11 — Wiktionary Frequency Centroid Sweep

**Goal**: scale the universal language vocabulary from 20 hand-picked seed concepts to a
data-driven estimate of how many *distinct* concepts the most common words across 6
languages actually encode.

**Method**:
1. Pull the top-N lemmas by frequency from `wordfreq` (backed by Wiktionary/OPUS data)
2. Embed all words per language with LaBSE
3. For each English word, find its nearest neighbour in every other language (pivot strategy)
4. Compute a speaker-weighted centroid for each aligned concept tuple
5. Cluster centroids with HDBSCAN to collapse synonymy and find the true vocabulary size
6. Annotate each entry with a **gap score** (mean pairwise cosine distance across languages)
   — this becomes the translatability flag for the universal language

**Key question**: how many concepts do 5 000 frequent words per language actually
encode? The answer should land in 2 000–3 000 after collapsing synonymy.

In [ ]:
!pip install sentence-transformers wordfreq hdbscan umap-learn matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
import warnings
warnings.filterwarnings('ignore')

import torch
from sentence_transformers import SentenceTransformer
from wordfreq import top_n_list
from sklearn.metrics.pairwise import cosine_distances

# Force CPU to avoid CUDA kernel image compatibility errors on Kaggle.
# LaBSE on CPU handles 2000-word sweeps in ~2 min/language.
# Switch to DEVICE='cuda' only if torch and CUDA versions are confirmed compatible.
# Force CPU — avoids CUDA kernel image errors (torch/CUDA version mismatch on Kaggle).
DEVICE = 'cpu'
print(f'Device: {DEVICE}')

model = SentenceTransformer('LaBSE', device=DEVICE)
print('LaBSE loaded')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Speaker counts in millions — used for centroid weighting
LANGS = {
    'en': 1500,
    'es': 560,
    'fr': 280,
    'pt': 260,
    'de': 130,
    'ja': 125,
}

# How many top-frequency words to pull per language
# 2000 is fast for a first run; push to 5000 for production
VOCAB_SIZE = 2000

# Minimum cosine similarity required for a cross-lingual match to be trusted
# Below this we consider the concept untranslatable via nearest-neighbour
SIM_THRESHOLD = 0.55

# HDBSCAN: minimum cluster size — tunes how aggressively synonymy is collapsed
MIN_CLUSTER_SIZE = 6

print(f'Config: vocab_size={VOCAB_SIZE}, sim_threshold={SIM_THRESHOLD}, '
      f'min_cluster_size={MIN_CLUSTER_SIZE}')

In [ ]:
# ── Step 1: Pull frequency lists and embed ────────────────────────────────────
# wordfreq.top_n_list returns words already lemmatised and lowercased,
# ranked by frequency in large multilingual corpora (Wikipedia + OpenSubtitles + more).

word_lists  = {}
emb_dict    = {}

for lang in LANGS:
    print(f'  [{lang}] pulling top {VOCAB_SIZE} words...')
    raw = top_n_list(lang, VOCAB_SIZE)
    # deduplicate while preserving order
    seen = set()
    words = []
    for w in raw:
        w = w.strip()
        if w and w not in seen:
            seen.add(w)
            words.append(w)
    word_lists[lang] = words
    print(f'  [{lang}] embedding {len(words)} unique lemmas...')
    embs = model.encode(words, normalize_embeddings=True, batch_size=128,
                        show_progress_bar=False)
    emb_dict[lang] = embs
    print(f'  [{lang}] done — shape {embs.shape}')

print('\nAll languages embedded.')

In [ ]:
# ── Step 2: Pivot alignment (English as hub) ──────────────────────────────────
# For each English word, find the nearest neighbour in every other language
# via a matrix multiply. This is O(|en| × |lang|) per language — fast on GPU.
#
# Why pivot on English? English embeddings sit closest to the LaBSE centroid
# (confirmed in Exp 6 & 7). Using it as hub minimises alignment errors.

en_words = word_lists['en']
en_embs  = emb_dict['en']          # (N_en, 768)

# For each non-English language: best match index and similarity for each EN word
align_idx  = {}   # lang → (N_en,) int array
align_sim  = {}   # lang → (N_en,) float array

for lang in LANGS:
    if lang == 'en':
        continue
    print(f'  Aligning EN → {lang} ({len(word_lists[lang])} words)...')
    # sim_matrix shape: (N_en, N_lang)
    sim_matrix = en_embs @ emb_dict[lang].T
    align_idx[lang]  = np.argmax(sim_matrix,  axis=1)   # best match per EN word
    align_sim[lang]  = np.max(sim_matrix,     axis=1)   # best similarity per EN word

print('Alignment complete.')

In [ ]:
# ── Step 3: Build aligned concept tuples ──────────────────────────────────────
# Only keep concepts where ALL languages have a match above SIM_THRESHOLD.
# This is strict — it filters out rare/technical words that don't translate cleanly.
# Those filtered-out words are themselves interesting (high-gap candidates).

aligned_concepts = []    # list of dicts: {lang: word}
aligned_centroids = []   # speaker-weighted centroid per concept
gap_scores = []          # mean pairwise cosine distance

skipped_weak  = 0
skipped_error = 0

for i, en_word in enumerate(en_words):
    concept = {'en': en_word}
    vecs = [en_embs[i] * LANGS['en']]     # weighted sum accumulator
    raw_vecs = [en_embs[i]]               # for gap score (unweighted)
    valid = True

    for lang in LANGS:
        if lang == 'en':
            continue
        sim = float(align_sim[lang][i])
        if sim < SIM_THRESHOLD:
            valid = False
            skipped_weak += 1
            break
        j    = int(align_idx[lang][i])
        word = word_lists[lang][j]
        concept[lang] = word
        lang_emb = emb_dict[lang][j]
        vecs.append(lang_emb * LANGS[lang])
        raw_vecs.append(lang_emb)

    if not valid:
        continue

    # Speaker-weighted centroid
    centroid = np.sum(vecs, axis=0)
    norm = np.linalg.norm(centroid)
    if norm < 1e-9:
        skipped_error += 1
        continue
    centroid = centroid / norm
    aligned_centroids.append(centroid)
    aligned_concepts.append(concept)

    # Gap score: mean pairwise cosine distance
    rv = np.array(raw_vecs)              # (n_langs, 768)
    sim_mat = rv @ rv.T
    dist_mat = 1.0 - sim_mat
    triu_idx = np.triu_indices(len(rv), k=1)
    gap = float(dist_mat[triu_idx].mean())
    gap_scores.append(gap)

centroid_matrix = np.array(aligned_centroids)

print(f'Concepts after alignment filter: {len(aligned_concepts)}')
print(f'  Skipped (weak alignment): {skipped_weak}')
print(f'  Skipped (numerical error): {skipped_error}')
print(f'  Centroid matrix shape: {centroid_matrix.shape}')

In [ ]:
# ── Step 4: HDBSCAN clustering — collapse synonymy ────────────────────────────
# Two concepts that align to the same region of embedding space are synonymous.
# HDBSCAN groups them. Each cluster = one entry in the universal vocabulary.
# Unclustered points (label = -1) = unique concepts with no close synonym.
#
# We use Euclidean distance on the centroid matrix because HDBSCAN is most
# stable with Euclidean. For normalised unit vectors, Euclidean distance is a
# monotone transform of cosine distance, so the clusters are equivalent.

print('Running HDBSCAN...')
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=2,
    metric='euclidean',
    cluster_selection_method='eom',
)
cluster_labels = clusterer.fit_predict(centroid_matrix)

n_clusters   = int((cluster_labels >= 0).sum() > 0 and cluster_labels.max() + 1)
n_noise      = int((cluster_labels == -1).sum())
# vocabulary estimate: one entry per cluster + one entry per noise point
vocab_est = n_clusters + n_noise

print(f'\n━━━ HDBSCAN Results ━━━')
print(f'  Clusters (synonymy groups):   {n_clusters}')
print(f'  Unclustered unique concepts:  {n_noise}')
print(f'  ──────────────────────────────')
print(f'  Estimated vocabulary size:    {vocab_est}')
print(f'  Input concepts (after filter): {len(aligned_concepts)}')
print(f'  Compression ratio:            {len(aligned_concepts)/max(vocab_est,1):.2f}x')

In [ ]:
# ── Step 5: Build results DataFrame ──────────────────────────────────────────
results_df = pd.DataFrame(aligned_concepts)
results_df['gap_score']   = gap_scores
results_df['cluster_id']  = cluster_labels

# Per-cluster representative: entry with highest mean similarity to cluster centroid
def cluster_representative(sub_df, centroid_mat):
    """Return the index of the concept closest to its cluster centroid."""
    c_ids = sub_df.index.tolist()
    vecs  = centroid_mat[c_ids]
    c     = vecs.mean(axis=0)
    c    /= np.linalg.norm(c)
    sims  = vecs @ c
    return c_ids[int(np.argmax(sims))]

rep_indices = set()
for cid in range(n_clusters):
    sub = results_df[results_df['cluster_id'] == cid]
    if len(sub) > 0:
        rep_idx = cluster_representative(sub, centroid_matrix)
        rep_indices.add(rep_idx)
# all noise points are their own representatives
for idx in results_df[results_df['cluster_id'] == -1].index:
    rep_indices.add(idx)

results_df['is_representative'] = results_df.index.isin(rep_indices)

# Summary statistics
print('Top 20 highest-gap concepts (hardest to translate):')
top_gap = results_df.sort_values('gap_score', ascending=False).head(20)
print(top_gap[['en', 'es', 'fr', 'de', 'gap_score', 'cluster_id']].to_string(index=False))

print('\nTop 10 lowest-gap concepts (most universal):')
bottom_gap = results_df[results_df['is_representative']].sort_values('gap_score').head(10)
print(bottom_gap[['en', 'es', 'fr', 'de', 'gap_score']].to_string(index=False))

## Visualisation

In [ ]:
# ── Figure 1: Vocabulary compression ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel A: gap score distribution
ax = axes[0]
ax.hist(results_df['gap_score'], bins=60, color='#378ADD', edgecolor='white', linewidth=0.3)
ax.axvline(0.15, color='#1D9E75', linestyle='--', label='low gap (universal)')
ax.axvline(0.30, color='#E24B4A', linestyle='--', label='high gap (untranslatable)')
ax.set_xlabel('Gap score (mean pairwise cosine distance)')
ax.set_ylabel('Number of concepts')
ax.set_title('Distribution of concept gap scores\nacross aligned vocabulary')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Panel B: cluster size distribution (how many synonyms per cluster)
ax = axes[1]
counts = results_df[results_df['cluster_id'] >= 0].groupby('cluster_id').size()
ax.hist(counts, bins=30, color='#1D9E75', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Cluster size (synonyms collapsed)')
ax.set_ylabel('Number of clusters')
ax.set_title(f'Synonymy cluster sizes\n({n_clusters} clusters, {n_noise} singletons)')
ax.grid(True, alpha=0.2)

# Panel C: vocabulary funnel
ax = axes[2]
stages = [
    f'Raw words\n({VOCAB_SIZE}/lang × {len(LANGS)} langs)',
    f'After\nalignment filter\n({len(aligned_concepts)})',
    f'After\nsynonymy collapse\n({vocab_est})',
]
values = [VOCAB_SIZE * len(LANGS), len(aligned_concepts), vocab_est]
colors = ['#E24B4A', '#378ADD', '#1D9E75']
bars = ax.barh(range(3), values, color=colors, height=0.5)
ax.set_yticks(range(3))
ax.set_yticklabels(stages, fontsize=9)
ax.set_xlabel('Number of entries')
ax.set_title('Vocabulary compression funnel\nfrom raw words to universal concepts')
for bar, v in zip(bars, values):
    ax.text(v + 20, bar.get_y() + bar.get_height()/2,
            f'{v:,}', va='center', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.2, axis='x')

plt.suptitle('Experiment 11 — Wiktionary Frequency Centroid Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('exp11_vocabulary_compression.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey result: estimated universal vocabulary size from'
      f' {VOCAB_SIZE} words × {len(LANGS)} languages = {vocab_est} distinct concepts')

In [ ]:
# ── Figure 2: Gap score by language ──────────────────────────────────────────
# For each concept, which language is most isolated?
# (i.e., whose translation lands furthest from the centroid)

print('Computing per-language isolation...')

lang_isolation = {lang: [] for lang in LANGS}

for i, concept in enumerate(aligned_concepts):
    centroid = centroid_matrix[i]
    for lang in LANGS:
        word_emb = None
        if lang == 'en':
            idx = en_words.index(concept['en'])
            word_emb = en_embs[idx]
        else:
            j = int(align_idx[lang][en_words.index(concept['en'])])
            word_emb = emb_dict[lang][j]
        dist = float(1 - np.dot(word_emb, centroid))
        lang_isolation[lang].append(dist)

fig, ax = plt.subplots(figsize=(9, 5))
lang_medians = {lang: np.median(dists) for lang, dists in lang_isolation.items()}
sorted_langs = sorted(lang_medians.items(), key=lambda x: x[1])
labels = [l for l, _ in sorted_langs]
data   = [lang_isolation[l] for l in labels]

bp = ax.boxplot(data, labels=labels, patch_artist=True, medianprops={'color': 'black', 'linewidth': 2})
pal = ['#1D9E75','#378ADD','#EF9F27','#D4537E','#7F77DD','#E24B4A']
for patch, color in zip(bp['boxes'], pal):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Language')
ax.set_ylabel('Cosine distance to universal centroid')
ax.set_title('Per-language isolation from universal concept centroid\n'
             '(lower = language sits closer to the universal representation)')
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('exp11_language_isolation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nLanguage centrality ranking (most → least central):')
for lang, med in sorted_langs:
    print(f'  {lang}: median isolation = {med:.4f}')

In [ ]:
# ── Export vocabulary table ───────────────────────────────────────────────────
# The representative concepts form the draft universal vocabulary.
# Save as CSV so Experiment 10 (grammatical frames) can consume it directly.

vocab_df = results_df[results_df['is_representative']].copy()
vocab_df = vocab_df.sort_values('gap_score')
vocab_df['translatability'] = pd.cut(
    vocab_df['gap_score'],
    bins=[-np.inf, 0.10, 0.20, 0.30, np.inf],
    labels=['universal', 'easy', 'hard', 'untranslatable']
)

vocab_df.to_csv('exp11_universal_vocab.csv', index=False)
print(f'Saved {len(vocab_df)} representative concepts to exp11_universal_vocab.csv')
print('\nTranslatability breakdown:')
print(vocab_df['translatability'].value_counts())

print('\nSample entries per tier:')
for tier in ['universal', 'easy', 'hard', 'untranslatable']:
    sub = vocab_df[vocab_df['translatability'] == tier].head(3)
    if len(sub):
        print(f'\n  [{tier}]')
        print(sub[['en', 'es', 'fr', 'de', 'ja', 'gap_score']].to_string(index=False))

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('━━━ Experiment 11 Summary ━━━')
print(f'  Input:         {VOCAB_SIZE} words × {len(LANGS)} languages = {VOCAB_SIZE*len(LANGS):,} raw tokens')
print(f'  After filter:  {len(aligned_concepts)} cross-linguistically aligned concepts')
print(f'  After cluster: {vocab_est} distinct universal concepts')
print(f'  Compression:   {VOCAB_SIZE*len(LANGS)/vocab_est:.1f}x redundancy removed')
print()
print('  Vocabulary tiers:')
for tier, count in vocab_df['translatability'].value_counts().items():
    pct = 100 * count / len(vocab_df)
    print(f'    {tier:<18} {count:>5}  ({pct:.1f}%)')
print()
print('  Next step → Experiment 10: feed vocab_df into grammatical frame classifier')
print('  Next step → Experiment 15: run phoneme centroid mapping on vocab_df')